# Tarefa Spark: lectura, transformación e escritura

Esta tarefa emprega o clúster e JupyterLab descritos no [contorno de traballo](../02.construcion-escenarios.md#contorno-de-traballo-do-curso) e os métodos do [tema de Spark](../06.spark.md). A sesión xa dispón da configuración necesaria. Os comandos de subida e listado traballáronse previamente no [tema de HDFS](../05.HDFS.md).

## Preparación

Copia [data/penguins.csv](../data/penguins.csv) ao cartafol `spark-notebooks/data` do host. Desde o terminal de JupyterLab, sube o ficheiro ao teu directorio HDFS:

```bash
hdfs dfs -mkdir -p /tmp/teunome/spark/datos
hdfs dfs -put /home/hadoop/work/data/penguins.csv /tmp/teunome/spark/datos/
```

Substitúe `teunome` polo teu identificador. Os exemplos Python execútanse nun notebook.

Para a parte JDBC, inicia tamén PostgreSQL desde PowerShell, no cartafol `big-data-cluster`:

```powershell
docker compose -f compose.base.yml -f compose.postgres.yml up -d
docker compose -f compose.base.yml -f compose.postgres.yml ps
```

Agarda a que `postgres-serving` apareza como **healthy**. Desde Spark, a URL é `jdbc:postgresql://postgres-serving:5432/gold_serving`, co usuario **`gold`**, o contrasinal **`gold`** e o driver **`org.postgresql.Driver`**. Emprega a imaxe do clúster que xa inclúe o driver JDBC. O exemplo guiado está no [laboratorio de lectura e escritura](../labs/lab-02-spark-lectura-escritura.ipynb).

## Organización do notebook

O [laboratorio meteorolóxico](../labs/lab-02-spark-lectura-escritura.ipynb) serve de modelo. Nesta tarefa empregarás Penguins, outras columnas de filtrado e agrupación, e os teus propios destinos. Completa as celas de traballo e conserva as saídas; non se piden explicacións escritas.

### Crear a sesión

Executa a creación da sesión nunha cela independente das rutas:


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("tarefa-penguins")
    .getOrCreate()
)


### Definir as rutas e o nome da táboa

Nunha segunda cela, define os destinos. Usa o mesmo identificador que nos comandos de subida a HDFS:


In [ ]:
usuario = "teunome"  # Substitúe polo teu identificador.
base = f"/tmp/{usuario}/spark"
entrada = f"{base}/datos/penguins.csv"
ruta_parquet = f"{base}/limpos_parquet"
taboa_pg = f"public.penguins_{usuario}"

print("Entrada HDFS:", entrada)
print("Saída Parquet:", ruta_parquet)
print("Táboa PostgreSQL:", taboa_pg)


## Exercicios

Para os exercicios 2 e 3 podes rexistrar os DataFrames como vistas temporais e empregar `spark.sql()`, seguindo a [introdución a Spark SQL](../06.spark.md#55-consultar-e-transformar-datos-con-spark-sql). As consultas devolven DataFrames que podes gardar en Parquet ou mediante JDBC.

### 1. Lectura e esquema (2 puntos).

Coa sesión xa creada, le o CSV con cabeceira e inferencia de tipos, mostra o esquema e conta as filas. O ficheiro ten 344 observacións.


In [ ]:
# Le entrada con cabeceira e inferencia de tipos.
# Mostra o esquema e o número de filas.


### 2. Calidade dos datos (1 punto).

Conta os nulos por columna. Conserva só as filas con `species`, `bill_length_mm` e `body_mass_g` informadas. Compara o reconto antes e despois; non elimines unha fila só por ter `sex` nulo.


In [ ]:
# Conta os nulos por columna.
# Crea df_limpo coas tres columnas obrigatorias informadas.
# Mostra os recontos antes e despois.


### 3. Consulta (1 punto).

A partir dos datos limpos do exercicio 2, calcula por especie o número de observacións e a media de `body_mass_g`. Ordena o resultado por especie.


In [ ]:
# Consulta os datos limpos: reconto e media de masa por especie.
# Mostra o resultado ordenado por especie.


### 4. Escritura e lectura (2 puntos).

Garda os datos limpos en Parquet nunha ruta propia baixo `/tmp/teunome/spark`. Volve lelos e comproba o esquema e o número de filas.


In [ ]:
# Garda df_limpo en ruta_parquet e volve lelo.
# Mostra o esquema e comproba o reconto.


### 5. Ficheiros e formatos (2 puntos).

Consulta o resultado con `hdfs dfs -ls`. Mostra xuntos o esquema recuperado de Parquet e o do CSV para comprobar os nomes e tipos das columnas.


In [ ]:
# Executa hdfs dfs -ls sobre ruta_parquet e conserva a saída.
# Mostra seguidos os esquemas do CSV e do Parquet.


### 6. Escritura e lectura JDBC (2 puntos).

Garda os datos limpos do exercicio 2 nunha táboa propia de PostgreSQL mediante `df.write.jdbc`, con modo `errorifexists` e un máximo dunha conexión simultánea (`numPartitions=1`). Emprega o esquema `public` e o nome `penguins_teunome`, substituíndo `teunome` polo teu identificador. Mostra o nome da táboa creada. **(1 punto)** Recupera a táboa mediante `spark.read.jdbc`, mostra o esquema e cinco filas, e comproba que o reconto coincide co dos datos limpos. **(1 punto)**


In [ ]:
# Escribe df_limpo en taboa_pg mediante JDBC.
# Recupera a táboa, mostra o esquema e cinco filas, e comproba o reconto.


Para repetir a escritura na túa ruta HDFS podes empregar `overwrite`. Non escribas sobre a ruta do CSV orixinal. Para repetir a parte JDBC podes empregar `overwrite` unicamente sobre a túa táboa de prácticas: este modo pode eliminar e recrear a táboa existente.

## Entrega

Notebook `apelido1_apelido2_nome_spark.ipynb` co código e os resultados. Inclúe a saída do listado HDFS, o nome da táboa PostgreSQL e os resultados da lectura e da comprobación JDBC. A tarefa suma 10 puntos e admite puntuación parcial segundo a corrección do código e dos resultados.

Ao rematar as comprobacións, pecha a sesión nunha cela final:


In [ ]:
spark.stop()
